# 04 — LifeSnaps Fitbit Subset Analysis

Analyzes relationships between daily activity, sleep, and heart rate using
the normalized Parquet subset produced by `scripts/build_lifesnaps_subset.py`.

**Research questions:**
1. Does more activity correlate with better sleep?
2. Does poor sleep correlate with higher next-day resting heart rate?
3. Are active minutes more useful than total steps?
4. Which users or metrics have the most missing data?
5. What problems appear when trying to unify wearable data across tables?

In [ ]:
from pathlib import Path
import json

import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted")

SUBSET_DIR = Path("../data/subset")
assert SUBSET_DIR.exists(), f"Subset not found at {SUBSET_DIR.resolve()}. Run scripts/build_lifesnaps_subset.py first."

## Load Data & Inspect Manifest

In [ ]:
manifest_path = SUBSET_DIR / "manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print(f"Subset created: {manifest['created_at']}")
    print(f"Users: {manifest['n_users']}, Date range: {manifest['date_range']}")
    print(f"Total size: {manifest['total_size_mb']:.1f} MB")
    print("\nTables:")
    for name, info in manifest["tables"].items():
        print(f"  {name}: {info['rows']:,} rows, {info['size_mb']:.1f} MB")
else:
    print("No manifest found — loading files directly.")

In [ ]:
daily = pl.read_parquet(SUBSET_DIR / "unified_daily_metrics.parquet")
sleep = pl.read_parquet(SUBSET_DIR / "unified_sleep.parquet")
hr = pl.read_parquet(SUBSET_DIR / "unified_heart_rate.parquet")

print(f"Daily metrics: {daily.shape}")
print(daily.head(3))
print(f"\nSleep: {sleep.shape}")
print(sleep.head(3))
print(f"\nHeart rate: {hr.shape}")
print(hr.head(3))

## Q4: Missingness Analysis

Which users and metrics have the most missing data? This is critical for
understanding data unification challenges.

In [ ]:
# Per-column null rates in daily metrics
metric_cols = ["steps", "calories", "sedentary_minutes",
               "lightly_active_minutes", "fairly_active_minutes",
               "very_active_minutes", "active_minutes"]

null_rates = daily.select(
    [pl.col(c).is_null().mean().alias(c) for c in metric_cols if c in daily.columns]
)
print("Daily metrics null rates:")
print(null_rates)

In [ ]:
# Build a user x date presence matrix across all three tables
all_users = sorted(set(
    daily["user_id"].unique().to_list() +
    sleep["user_id"].unique().to_list() +
    hr["user_id"].unique().to_list()
))

# Count days each user has data in each table
daily_date_col = "date" if "date" in daily.columns else daily.columns[1]
sleep_date_col = "sleep_date" if "sleep_date" in sleep.columns else sleep.columns[1]
hr_date_col = "date" if "date" in hr.columns else hr.columns[1]

user_coverage = []
for uid in all_users:
    d_days = daily.filter(pl.col("user_id") == uid).select(daily_date_col).n_unique()
    s_days = sleep.filter(pl.col("user_id") == uid).select(sleep_date_col).n_unique()
    h_days = hr.filter(pl.col("user_id") == uid).select(hr_date_col).n_unique()
    user_coverage.append({"user_id": str(uid)[:8], "daily_days": d_days, "sleep_days": s_days, "hr_days": h_days})

coverage_df = pl.DataFrame(user_coverage)
print(coverage_df.sort("daily_days", descending=True))

In [ ]:
# Missingness heatmap: user x metric presence
fig, ax = plt.subplots(figsize=(8, max(6, len(all_users) * 0.3)))

heatmap_data = coverage_df.select(["daily_days", "sleep_days", "hr_days"]).to_numpy()
user_labels = coverage_df["user_id"].to_list()

sns.heatmap(
    heatmap_data,
    xticklabels=["Daily Activity", "Sleep", "Heart Rate"],
    yticklabels=user_labels,
    annot=True, fmt=".0f", cmap="YlOrRd_r",
    ax=ax
)
ax.set_title("Days of Data per User per Metric")
ax.set_ylabel("User (truncated ID)")
plt.tight_layout()
plt.show()

## Join Tables for Cross-Metric Analysis

Join sleep, daily activity, and heart rate by user + date to enable
correlation analysis.

In [ ]:
# Standardize date columns to string for joining
daily_j = daily.with_columns(pl.col(daily_date_col).cast(pl.Utf8).alias("join_date"))
sleep_j = sleep.with_columns(pl.col(sleep_date_col).cast(pl.Utf8).alias("join_date"))
hr_j = hr.with_columns(pl.col(hr_date_col).cast(pl.Utf8).alias("join_date"))

# Join daily + sleep
daily_sleep = daily_j.join(
    sleep_j.select(["user_id", "join_date", "duration_minutes", "efficiency"]),
    on=["user_id", "join_date"],
    how="inner",
)
print(f"Daily + Sleep joined: {daily_sleep.shape[0]} rows")

# Join daily + HR
daily_hr = daily_j.join(
    hr_j.select(["user_id", "join_date", "resting_hr", "avg_hr"]),
    on=["user_id", "join_date"],
    how="inner",
)
print(f"Daily + HR joined: {daily_hr.shape[0]} rows")

# Full three-way join
full = daily_sleep.join(
    hr_j.select(["user_id", "join_date", "resting_hr", "avg_hr"]),
    on=["user_id", "join_date"],
    how="left",
)
print(f"Full three-way join: {full.shape[0]} rows")

## Q1: Does More Activity Correlate with Better Sleep?

In [ ]:
# Filter to rows with valid data
q1_df = daily_sleep.filter(
    pl.col("steps").is_not_null() &
    pl.col("duration_minutes").is_not_null() &
    pl.col("active_minutes").is_not_null()
).to_pandas()

if len(q1_df) > 10:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Steps vs sleep duration
    axes[0].scatter(q1_df["steps"], q1_df["duration_minutes"], alpha=0.3, s=10)
    r_steps, p_steps = stats.pearsonr(q1_df["steps"].dropna(), q1_df["duration_minutes"].dropna())
    axes[0].set_xlabel("Steps")
    axes[0].set_ylabel("Sleep Duration (min)")
    axes[0].set_title(f"Steps vs Sleep Duration\nr={r_steps:.3f}, p={p_steps:.3e}")

    # Active minutes vs sleep duration
    axes[1].scatter(q1_df["active_minutes"], q1_df["duration_minutes"], alpha=0.3, s=10)
    r_active, p_active = stats.pearsonr(q1_df["active_minutes"].dropna(), q1_df["duration_minutes"].dropna())
    axes[1].set_xlabel("Active Minutes")
    axes[1].set_ylabel("Sleep Duration (min)")
    axes[1].set_title(f"Active Minutes vs Sleep Duration\nr={r_active:.3f}, p={p_active:.3e}")

    plt.tight_layout()
    plt.show()

    # Also check sleep efficiency
    q1_eff = daily_sleep.filter(
        pl.col("steps").is_not_null() & pl.col("efficiency").is_not_null()
    ).to_pandas()
    if len(q1_eff) > 10:
        r_eff, p_eff = stats.pearsonr(q1_eff["steps"], q1_eff["efficiency"])
        print(f"\nSteps vs Sleep Efficiency: r={r_eff:.3f}, p={p_eff:.3e}")
        r_act_eff, p_act_eff = stats.pearsonr(q1_eff["active_minutes"], q1_eff["efficiency"])
        print(f"Active Minutes vs Sleep Efficiency: r={r_act_eff:.3f}, p={p_act_eff:.3e}")
else:
    print(f"Insufficient data for Q1 analysis ({len(q1_df)} rows after filtering)")

## Q2: Does Poor Sleep Correlate with Higher Next-Day Resting HR?

Lag-1 analysis: join tonight's sleep with tomorrow's resting heart rate.

In [ ]:
# Create lagged join: sleep on date D -> HR on date D+1
sleep_for_lag = sleep_j.with_columns(
    (pl.col(sleep_date_col).cast(pl.Date) + pl.duration(days=1)).cast(pl.Utf8).alias("next_day")
)

q2_df = sleep_for_lag.join(
    hr_j.select(["user_id", "join_date", "resting_hr"]),
    left_on=["user_id", "next_day"],
    right_on=["user_id", "join_date"],
    how="inner",
).filter(
    pl.col("duration_minutes").is_not_null() & pl.col("resting_hr").is_not_null()
).to_pandas()

if len(q2_df) > 10:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Sleep duration vs next-day resting HR
    axes[0].scatter(q2_df["duration_minutes"], q2_df["resting_hr"], alpha=0.3, s=10)
    r, p = stats.pearsonr(q2_df["duration_minutes"], q2_df["resting_hr"])
    axes[0].set_xlabel("Sleep Duration (min)")
    axes[0].set_ylabel("Next-Day Resting HR (bpm)")
    axes[0].set_title(f"Sleep Duration → Next-Day Resting HR\nr={r:.3f}, p={p:.3e}")

    # Sleep efficiency vs next-day resting HR
    q2_eff = q2_df.dropna(subset=["efficiency"])
    if len(q2_eff) > 10:
        axes[1].scatter(q2_eff["efficiency"], q2_eff["resting_hr"], alpha=0.3, s=10)
        r2, p2 = stats.pearsonr(q2_eff["efficiency"], q2_eff["resting_hr"])
        axes[1].set_xlabel("Sleep Efficiency")
        axes[1].set_ylabel("Next-Day Resting HR (bpm)")
        axes[1].set_title(f"Sleep Efficiency → Next-Day Resting HR\nr={r2:.3f}, p={p2:.3e}")
    else:
        axes[1].text(0.5, 0.5, "Insufficient efficiency data", ha="center", va="center")

    plt.tight_layout()
    plt.show()
else:
    print(f"Insufficient data for lagged HR analysis ({len(q2_df)} rows)")

## Q3: Are Active Minutes More Useful Than Total Steps?

Compare predictive power of steps vs active_minutes for sleep outcomes
using Spearman correlation and partial correlation.

In [ ]:
q3_df = daily_sleep.filter(
    pl.col("steps").is_not_null() &
    pl.col("active_minutes").is_not_null() &
    pl.col("duration_minutes").is_not_null()
).to_pandas()

if len(q3_df) > 10:
    # Spearman (rank-based, better for non-linear relationships)
    rho_steps, p_steps = stats.spearmanr(q3_df["steps"], q3_df["duration_minutes"])
    rho_active, p_active = stats.spearmanr(q3_df["active_minutes"], q3_df["duration_minutes"])

    print("Spearman correlations with sleep duration:")
    print(f"  Steps:          rho={rho_steps:.4f}, p={p_steps:.3e}")
    print(f"  Active minutes: rho={rho_active:.4f}, p={p_active:.3e}")
    print(f"  Winner: {'Active minutes' if abs(rho_active) > abs(rho_steps) else 'Steps'}")

    # If efficiency is available
    q3_eff = daily_sleep.filter(
        pl.col("steps").is_not_null() &
        pl.col("active_minutes").is_not_null() &
        pl.col("efficiency").is_not_null()
    ).to_pandas()
    if len(q3_eff) > 10:
        rho_steps_eff, _ = stats.spearmanr(q3_eff["steps"], q3_eff["efficiency"])
        rho_active_eff, _ = stats.spearmanr(q3_eff["active_minutes"], q3_eff["efficiency"])
        print(f"\nSpearman correlations with sleep efficiency:")
        print(f"  Steps:          rho={rho_steps_eff:.4f}")
        print(f"  Active minutes: rho={rho_active_eff:.4f}")

    # Visualize the comparison
    fig, ax = plt.subplots(figsize=(8, 5))
    metrics = ["Steps", "Active Min"]
    correlations = [rho_steps, rho_active]
    colors = ["steelblue", "coral"]
    bars = ax.bar(metrics, correlations, color=colors)
    ax.set_ylabel("Spearman rho with Sleep Duration")
    ax.set_title("Steps vs Active Minutes: Which Predicts Sleep Better?")
    ax.axhline(0, color="gray", linestyle="--", linewidth=0.5)
    for bar, val in zip(bars, correlations):
        ax.text(bar.get_x() + bar.get_width() / 2, val, f"{val:.3f}",
                ha="center", va="bottom" if val > 0 else "top")
    plt.tight_layout()
    plt.show()
else:
    print(f"Insufficient data for Q3 ({len(q3_df)} rows)")

## Q5: Data Unification Challenges

Document the practical problems encountered when joining wearable data
across tables.

In [ ]:
# Challenge 1: Date alignment
# Sleep date can refer to the night of or the morning of depending on provider
daily_dates = set(daily_j["join_date"].unique().to_list())
sleep_dates = set(sleep_j["join_date"].unique().to_list())
hr_dates = set(hr_j["join_date"].unique().to_list())

print("Date coverage overlap:")
print(f"  Daily ∩ Sleep: {len(daily_dates & sleep_dates)} / {len(daily_dates | sleep_dates)} dates")
print(f"  Daily ∩ HR:    {len(daily_dates & hr_dates)} / {len(daily_dates | hr_dates)} dates")
print(f"  Sleep ∩ HR:    {len(sleep_dates & hr_dates)} / {len(sleep_dates | hr_dates)} dates")
print(f"  All three:     {len(daily_dates & sleep_dates & hr_dates)} dates")

In [ ]:
# Challenge 2: Duplicate records
daily_dupes = daily.group_by(["user_id", daily_date_col]).len().filter(pl.col("len") > 1)
sleep_dupes = sleep.group_by(["user_id", sleep_date_col]).len().filter(pl.col("len") > 1)

print(f"\nDuplicate user-date pairs:")
print(f"  Daily metrics: {len(daily_dupes)} duplicates")
print(f"  Sleep: {len(sleep_dupes)} duplicates")

if len(sleep_dupes) > 0:
    print("  (Multiple sleep records per night is common — naps, split sleep)")

In [ ]:
# Challenge 3: Schema inconsistencies and type issues
print("\nSchema report:")
for name, df in [("daily", daily), ("sleep", sleep), ("hr", hr)]:
    print(f"\n  {name}:")
    for col in df.columns:
        null_pct = df[col].is_null().mean() * 100
        print(f"    {col:30s} {str(df[col].dtype):12s} {null_pct:5.1f}% null")

In [ ]:
# Challenge 4: Temporal gaps — visualize data availability over time
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

for idx, (name, df, date_col) in enumerate([
    ("Daily Activity", daily, daily_date_col),
    ("Sleep", sleep, sleep_date_col),
    ("Heart Rate", hr, hr_date_col),
]):
    date_counts = df.group_by(date_col).len().sort(date_col)
    dates = date_counts[date_col].to_list()
    counts = date_counts["len"].to_list()
    axes[idx].bar(range(len(dates)), counts, width=1.0, alpha=0.7)
    axes[idx].set_ylabel("Records")
    axes[idx].set_title(f"{name} — records per date")
    if len(dates) > 0:
        n_ticks = min(10, len(dates))
        tick_positions = np.linspace(0, len(dates) - 1, n_ticks, dtype=int)
        axes[idx].set_xticks(tick_positions)
        axes[idx].set_xticklabels([str(dates[i])[:10] for i in tick_positions], rotation=45, ha="right")

plt.suptitle("Temporal Data Availability", y=1.01)
plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)

print(f"\nDataset: {manifest.get('n_users', '?')} users, {manifest.get('date_range', '?')}")
print(f"Total subset size: {manifest.get('total_size_mb', '?')} MB")

print(f"\nJoinability:")
print(f"  Daily + Sleep inner join: {daily_sleep.shape[0]} rows")
print(f"  Daily + HR inner join: {daily_hr.shape[0]} rows")
print(f"  Full three-way: {full.shape[0]} rows")

join_rate_sleep = daily_sleep.shape[0] / max(daily.shape[0], 1) * 100
join_rate_hr = daily_hr.shape[0] / max(daily.shape[0], 1) * 100
print(f"  Sleep join rate: {join_rate_sleep:.1f}% of daily rows have matching sleep")
print(f"  HR join rate: {join_rate_hr:.1f}% of daily rows have matching HR")

print("\nKey findings:")
print("  (Populated after running the analysis above)")